# ap2111_analyze_data

## Reading an Organization ID and Analyzing the Corresponding Data

### Summary

This notebook reads an organization ID (OrgID) and retrieves the associated data either through the **data_loading module** or from a **CSV export**. The dataset is then subjected to multiple quality, plausibility, and consistency checks. Further analyses are performed for meter points, generators, and consumers, including visual evaluations.

### Notebook Structure

1. **Imports & Definitions**  
    Initialization of all required libraries, helper functions, and configuration parameters.
    
2. **Reading in Data**
    
    - **From Database:**  
        Uses the data_loading module to query the necessary tables based on the provided OrgID.
        
    - **From CSV:**  
        Provides an alternative pathway to load previously exported datasets.
        
3. **Basic Data Analyses**
    
    - **Validation Checks:** Ensures structural correctness of all key fields.
        
    - **Value Checks:** Reviews the ranges and plausibility of meter readings and metadata attributes.
        
    - **Time Change Detection:** Verifies whether daylight saving time (summer/winter time) transitions are not present.
        
    - **Consistency Check for mp_id:** Confirms that meter point identifiers match the metadata definitions.
        
4. **Further Analyses**
    
    - **Analysis of Generators and Consumers:** Breaks down energy production and consumption profiles.
        
    - **Graphical Representation:** Visualizes load curves, generation patterns, and other relevant metrics.
        
5. **Missing Timestamps (Completeness Check)**  
    Identifies gaps in the time series on meter-point level and evaluates overall data completeness.
    
6. **Aggregated EEG-Level Analysis**  
    Creates aggregated datasets per EEG and performs high-level analyses across the grouped data.
    

### Findings

- No transition between summer and winter time was detected; no adjustments are required.
    
- The dataset is up to date, containing only a few days-old records.
    
- All values appear plausible; no erroneous or extreme values were observed.
    
- Meter point IDs and metadata are consistent; no mismatches were found.
    
- The dataset is largely complete. While some meter points show missing timestamps, each EEG is complete as a whole.
    
- All measured values lie within acceptable and expected ranges.
    

### Open Tasks

- Perform the analyses for all EEGs to ensure full coverage.

Information on notation

* CC	Community Coverage	Self-coverage	kWh
* CP	Community Potential	Community share	kWh
* WMC	Weighted Measured Consumption	Measured consumption weighted by participation factor	kWh
* WMG	Weighted Measured Generation	Measured generation weighted by participation factor	kWh
* WSG	Weighted Surplus Generation	Residual surplus weighted by participation factor	kWh

### 01 Imports & Definitionen

In [ ]:
#imports
import numpy as np
import pandas as pd
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from datetime import datetime
from modules.data_loading import load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine, load_master_data

In [ ]:
use_csv = 0  # 1 = use CSV files, 0 = use database

#for database import
org_id = 12
time_start = datetime(2025, 1, 1)
time_end = datetime(2025, 9, 30)

#for csv import
path_to_local_data = "../../local_data/"

date = datetime.now()

### 02 Reading in data

#### 02.01 Reading from Databese

In [ ]:
if use_csv == 0:

    #definitions
    time_start = datetime(2025, 1, 1)
    time_end = datetime(2025, 9, 30)

    load_dotenv()
    ssh_host = os.getenv("SSH_HOST")
    ssh_port = int(os.getenv("SSH_PORT"))
    ssh_user = os.getenv("SSH_USER")
    ssh_pw = os.getenv("SSH_PASSWORD")

    postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
    postgres_port = int(os.getenv("POSTGRES_PORT"))

    with SSHTunnelForwarder(
        (ssh_host, ssh_port),
        ssh_username=ssh_user,
        ssh_password=ssh_pw,
        remote_bind_address=(ssh_host, postgres_port),
        local_bind_address=(postgres_server_ip, postgres_port)
    ) as tunnel:
        df=load_all_metering_points_in_energy_community_data(org_id=org_id, time_start=time_start, time_end=time_end, sql_engine=get_postgres_engine()
                                )

#### 02.02 from csv

In [ ]:
if use_csv == 1:

    df = pd.read_csv(path+org_csv)

In [ ]:
df.head()

In [ ]:
# copy dataframe and convert time column to datetime
org_df = df.copy()
org_df['time'] = pd.to_datetime(org_df['time']) #make datetime collumn

### 03 Analyzing data set

In [ ]:
#summary statistics
summary = pd.DataFrame({
        'dtype': org_df.dtypes,
        'missing': org_df.isna().sum(),
        'min': org_df.min(numeric_only=False),
        'max': org_df.max(numeric_only=False)
    })

print(summary)

In [ ]:
# check if values between 0 and 100
cols = ['wt_meas_cons', 'comm_pot', 'comm_cov', 'wt_meas_gen', 'wt_surp_gen']
all_ok = True

for col in cols:
    valid = df[col].dropna()
    if valid.between(0, 100).all():
        pass
    else:
        print(f"not all values in '{col}' between 0 and 100!")
        print(df.loc[~df[col].between(0, 100), [col]])
        all_ok = False

if all_ok:
    print("All values between 0 and 100!")  

In [ ]:
#unique values in collumns
for col in ["mp_id", "period_interval", "energy_direction"]:
    unique_vals = org_df[col].unique()
    print(f"\n {col} — {len(unique_vals)} unique values:")
    print(unique_vals)

In [ ]:
#Sample plot for distribution of 'wt_meas_gen'

plt.figure(figsize=(10, 6))
org_df["wt_meas_gen"].hist(bins=50)
plt.title("Distribution of values in 'wt_meas_gen'")
plt.xlabel("value")
plt.ylabel("frequency")
plt.grid(True)
plt.show()

check summer/winter time

In [ ]:
check_time = org_df.groupby(['mp_id', 'time']).size().loc[lambda x: x > 5]
if len(check_time) == 0:
    print("No duplicate time entries found!")
else:
    print("Duplicate time entries found:")
    print(check_time)

we have no change from summer to wintertime and so we don’t need any actions her

check if mp_id valid

In [ ]:
#load file with all org_id's and mp_ids
path = 'data/processed_data/'
org_csv = 'df_v_proj_org_mp.csv'
df_mp_id = pd.read_csv(path+org_csv)

#filter df by org_id
df_mp_id = df_mp_id[df_mp_id['org_id'] == org_id]

#df_mp_id.head()

In [ ]:
ids_df1 = set(org_df['mp_id'].unique())
ids_df2 = set(df_mp_id['mp_id'].unique())

# Check whether all IDs from df1 are also in df2
check_all = ids_df1.issubset(ids_df2)

print("All mp_id from data valide?", check_all)

#which IDs are missing:
if not check_all:
    inavlid_ids = ids_df1 - ids_df2
    print("invalid mp_id:", inavlid_ids)

### 04 further analysis

In [ ]:
# # Export the data as CSV files
# output_path = 'data/processed_data/'
# if not os.path.exists(output_path):
#     os.makedirs(output_path)    
# org_df.to_csv(output_path + f'org_df_org_{org_id}.csv', index=False)

In [ ]:
eeg_cleaned2.head()

In [ ]:
eeg_cleaned = org_df.copy()
eeg_cleaned.head()

analyses of generators and consumers

In [ ]:
# counts of metering points
consumer = eeg_cleaned[eeg_cleaned["energy_direction"] == "C"]
generators = eeg_cleaned[eeg_cleaned["energy_direction"] == "G"]

mp_cnt = len(org_df["mp_id"].unique())
cons_cnt = len(consumer["mp_id"].unique())
gen_cnt = len(generators["mp_id"].unique())

print(f"Sum of mp in EEG {org_id}: {mp_cnt}")
print(f"\t consumers: {cons_cnt} ({cons_cnt/mp_cnt*100:.1f}%)")
print(f"\t generators: {gen_cnt} ({gen_cnt/mp_cnt*100:.1f}%)")
print()

In [ ]:
#sort after time
generators = generators.sort_values("time")
generators.head()

In [ ]:
consumer = consumer.sort_values("time")
consumer.head()

graphical representation

In [ ]:
# Plot
fig = px.line(
    consumer,
    x="time",
    y="wt_meas_cons",
    title=f"wt_meas_cons over time"
)

fig.show()

In [ ]:
# filter for mp_id 
id = 4390
df_consumer = consumer[consumer['mp_id'] == id].sort_values('time')

# plot
fig = px.line(
    df_consumer,
    x="time",
    y="wt_meas_cons",
    title=f"wt_meas_cons over time for mp_id {id}"
)

fig.show()

In [ ]:
# plot
fig = px.line(
    generators,
    x="time",
    y="wt_meas_gen",
    title=f"wt_meas_gen over time"
)

fig.show()

In [ ]:
# filter for mp_id
id = 4401
df_generators = generators[generators['mp_id'] == id].sort_values('time')

# plot
fig = px.line(
    df_generators,
    x="time",
    y="wt_meas_gen",
    title=f"wt_meas_gen over time for mp_id {id}"
)

fig.show()

In [ ]:
# Summations
sum_eeg_cons = consumer["wt_meas_cons"].sum()
sum_eeg_gen = generators["wt_meas_gen"].sum()
sum_eeg_surp = generators["wt_surp_gen"].sum()
print(f"Sum consumations all mp: {sum_eeg_cons:.0f} kWh")
print(f"Total generation all mp: {sum_eeg_gen:.0f} kWh")
print(f"\tsurplus of all mp: {sum_eeg_surp:.0f} kWh -> {sum_eeg_surp/sum_eeg_gen*100:.0f}%")

In [ ]:
# Calculation of the number of unique counting points per day
eeg_cleaned["time_day"] = pd.to_datetime(eeg_cleaned["time"]).dt.floor("D")

# Count the number of unique org_ids for each individual day.
daily_counts = (
    eeg_cleaned.groupby(["time_day", "energy_direction"])["mp_id"]
    .nunique()
    .reset_index(name="unique_obj_count")
)

# If energy_direction is not set, it is replaced with 0.
daily_counts = (
    daily_counts
    .pivot(index="time_day", columns="energy_direction", values="unique_obj_count")
    .fillna(0)
    .reset_index()
)

In [ ]:
# plotly chart of count of metring point per day
fig = px.area(
    daily_counts,
    x="time_day",
    y=["C", "G"],  # Consumer + Generator
    title="Number of unique metering points per day (Consumer vs. Generator)",
    labels={
        "time_day": "Date",
        "value": "Number of unique metering points",
        "variable": "Energy direction",
    },
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text="Energy direction",
    template="plotly_white"
)

fig.show()

In [ ]:
# PRAMATER

# ====================================

date_start = "2025-06-18"
date_end = "2025-06-28"

# ====================================

In [ ]:
# Calculation of totals per counting point for entire time period
temp_timefiltered = eeg_cleaned[
    (eeg_cleaned["time"] > date_start) & (eeg_cleaned["time"] < date_end)
].copy()

# Total energy consumption (wt_meas_cons) and production (wt_meas_gen) for the entire period
total_sum = (
    temp_timefiltered.groupby(["mp_id", "energy_direction"], as_index=False)[["wt_meas_gen", "wt_meas_cons"]]
    .sum()
)

# prepaire for Plotly
total_sum_melted = total_sum.melt(
    id_vars=["mp_id", "energy_direction"],
    value_vars=["wt_meas_gen", "wt_meas_cons"],
    var_name="Typ",
    value_name="Sum"
)

total_sum_melted = total_sum_melted[total_sum_melted["Sum"] != 0].copy()


In [ ]:
# plots histplot of Summed Consumption for each single mp
daily_sum_c = total_sum_melted[total_sum_melted["energy_direction"] == "C"].copy()

# Histogramm for energy_direction = "C"
fig = px.histogram(
    daily_sum_c,
    x="Sum",
    color="Typ",
    nbins=50,
    marginal="box",        # Boxplot
    opacity=0.7,
    color_discrete_sequence=["#E66E46", "#EE715B"],
    title=f"Histogram of the sums per counting point in '{date_start}' - '{date_end}'\n(only Consumption - 'C')"
)

fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="measurement method",
    xaxis_title="Total consumption in kWh per metering point",
    yaxis_title="count"
)

fig.show()


In [ ]:
sum_eeg_cons = temp_timefiltered["wt_meas_cons"].sum()
sum_eeg_comm_cov = temp_timefiltered["comm_cov"].sum()
print(f"Sum all consumation all mp: {sum_eeg_cons:.2f} kWh (in period '{date_start}' - '{date_end}')")
print(f"\t of which covered by the EEG: {sum_eeg_comm_cov:.2f} kWh / {sum_eeg_comm_cov/sum_eeg_cons*100:.0f}%")

In [ ]:
# plots histplot of Summed Generation for each single mp
# "G" -> generators
daily_sum_g = total_sum_melted[total_sum_melted["energy_direction"] == "G"].copy()

#  Histogramm for energy_direction = "G" 
fig = px.histogram(
    daily_sum_g,
    x="Sum",
    color="Typ",
    nbins=50,
    marginal="box",        # Boxplotk
    opacity=0.7,
    color_discrete_sequence=["#FFD166", "#EF8A17"],  
    title=f"Histogram of the sums per counting point in '{date_start}' - '{date_end}' (only Generation - 'G')"
)


fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="measurement method",
    xaxis_title="Total production in kWh per feeder",
    yaxis_title="count"
)

fig.show()


In [ ]:
sum_eeg_gen = daily_sum_g["Sum"].sum()
print(f"Sum production all mp: {sum_eeg_gen:.4f} kwH (in period '{date_start}' - '{date_end}')")

analysis of the time series for gaps

### 05 missing timestamps

In [ ]:
# final list for gaps over eeg
gaps = []

# group for mp_id
for mp_id, group in eeg_cleaned.groupby("mp_id"):
    group = group.sort_values("time")

    # Time differences between consecutive points in time
    diffs = group["time"].diff().dropna()

    # max gap
    max_gap = diffs.max()

    # If gaps are not constant 
    if len(diffs.unique()) > 1:
        gaps.append({
            "mp_id": mp_id,
            "number_of_missing_values": len(group),
            "max_gap": max_gap,
            "deviating_distances": True
        })
    else:
        gaps.append({
            "mp_id": mp_id,
            "number_of_missing_values": len(group),
            "max_gap": max_gap,
            "deviating_distances": False
        })

# safe as df
gap_df = pd.DataFrame(gaps)
gap_df = gap_df.sort_values(by="number_of_missing_values", ascending=False)

if len(gap_df) == 0:
    print("No missing time entries found!")
else:   
    print(gap_df)

In [ ]:
#check missings over eeg
full_range = pd.date_range(
    start=eeg_cleaned['time'].min(),
    end=eeg_cleaned['time'].max(),
    freq='15T'
)

missing_times = full_range.difference(eeg_cleaned['time'])

if len(missing_times) == 0:
    print("No missing time entries found!")
else:   
    print(missing_times)

### 06 Analyse Aggregating data set per EEG

In [ ]:
cols_to_mean = ['wt_meas_cons', 'comm_pot', 'comm_cov', 'wt_meas_gen', 'wt_surp_gen']
df_agg = (
    eeg_cleaned.groupby('time')[cols_to_mean]
      .mean()
      .reset_index()
)

In [ ]:
df_agg.head(-1)

In [ ]:
summary = pd.DataFrame({
        'dtype': df_agg.dtypes,
        'missing': df_agg.isna().sum(),
        'min': df_agg.min(numeric_only=False),
        'max': df_agg.max(numeric_only=False)
    })

print(summary)

In [ ]:
# Plots
for c in cols_to_mean:

    fig = px.line(
        df_agg,
        x="time",
        y=c,
        title=f"{c} over time",
    )

    fig.show()

### 10 Save data as csv

In [ ]:
#pivot_df.to_csv('data/processed_data/aggregated_data.csv', index=False)